# EcoShield AI — Hafif Model Karşılaştırması

Bu notebook ortak split üzerinde dört hafif model adayını karşılaştırır:

1. Logistic Regression
2. Decision Tree
3. Small Random Forest
4. Small LightGBM

Model seçimi kesinleştirilmez ve threshold tuning yapılmaz. Karşılaştırma yalnızca validation splitinde, tanısal `0.50` eşiği ve eşikten bağımsız ROC-AUC/PR-AUC metrikleriyle yapılır. Test spliti belleğe yüklenmez veya değerlendirilmez.

In [9]:
%pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


## 1. Ayarlar ve paket kontrolü

In [10]:
QUICK_MODE = False
RANDOM_STATE = 42
DIAGNOSTIC_THRESHOLD = 0.50
INFERENCE_SAMPLE_SIZE = 10_000
INFERENCE_REPEATS = 5

import importlib.util
import json
import sys
from pathlib import Path

required = {
    "joblib": "joblib", "lightgbm": "lightgbm", "numpy": "numpy",
    "pandas": "pandas", "psutil": "psutil", "scipy": "scipy",
    "sklearn": "scikit-learn", "tqdm": "tqdm",
}
missing = [pip for module, pip in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ModuleNotFoundError("Eksik paketler: " + ", ".join(missing))

print("Ayarlar hazır. Test spliti bu notebookta yüklenmeyecek.")

Ayarlar hazır. Test spliti bu notebookta yüklenmeyecek.


## 2. Importlar ve ortak modül

Profil cache'i mevcutsa doğrudan yüklenir; yoksa Görev 1'de üretilen ortak Parquet'ten bir kez hazırlanıp saklanır.

In [11]:
import os
import time

import joblib
import numpy as np
import pandas as pd
import psutil
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "common_preprocessing.py").exists():
    candidate = NOTEBOOK_DIR / "notebooks"
    if (candidate / "common_preprocessing.py").exists():
        NOTEBOOK_DIR = candidate
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from common_preprocessing import (
    build_project_paths,
    find_project_root,
    get_or_create_profile_cache,
)

PROJECT_ROOT = find_project_root(Path.cwd())
PATHS = build_project_paths(PROJECT_ROOT)
MODEL_DIR = PROJECT_ROOT / "models" / "light"
PREDICTIONS_DIR = PATHS.outputs / "predictions"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print("Proje kökü:", PROJECT_ROOT)

Proje kökü: C:\Users\pc\Desktop\YZTA-Bootcamp-2026


In [12]:
import importlib
import inspect
import common_preprocessing as cp

cp = importlib.reload(cp)

print("Yüklenen dosya:", cp.__file__)
print("Fonksiyon:", inspect.signature(cp.get_or_create_profile_cache))

get_or_create_profile_cache = cp.get_or_create_profile_cache

Yüklenen dosya: c:\Users\pc\Desktop\YZTA-Bootcamp-2026\notebooks\common_preprocessing.py
Fonksiyon: (profile: 'str', project_root: 'Path | None' = None, *, quick_mode: 'bool' = False, force_rebuild: 'bool' = False, include_test: 'bool' = True) -> 'dict[str, Any]'


## 3. Train ve validation cache'lerini yükleme

`include_test=False` sayesinde test feature, target ve ID dizileri bu notebooka alınmaz.

In [13]:
linear_data = get_or_create_profile_cache(
    "logistic_regression", PROJECT_ROOT,
    quick_mode=QUICK_MODE, include_test=False,
)
tree_data = get_or_create_profile_cache(
    "random_forest", PROJECT_ROOT,
    quick_mode=QUICK_MODE, include_test=False,
)
lightgbm_data = get_or_create_profile_cache(
    "lightgbm", PROJECT_ROOT,
    quick_mode=QUICK_MODE, include_test=False,
)

assert "X_test" not in linear_data
assert "X_test" not in tree_data
assert "X_test" not in lightgbm_data
assert np.array_equal(linear_data["y_validation"], tree_data["y_validation"])
assert np.array_equal(linear_data["y_validation"], lightgbm_data["y_validation"])
assert np.array_equal(linear_data["id_validation"], tree_data["id_validation"])
assert np.array_equal(linear_data["id_validation"], lightgbm_data["id_validation"])

print("Linear:", linear_data["X_train"].shape, linear_data["X_validation"].shape)
print("Tree:", tree_data["X_train"].shape, tree_data["X_validation"].shape)
print("LightGBM:", lightgbm_data["X_train"].shape, lightgbm_data["X_validation"].shape)

linear cache bulunamadı; ortak Parquet'ten oluşturuluyor.

BAŞLADI: Ortak Parquet cache'ini yükleme
TAMAMLANDI: Ortak Parquet cache'ini yükleme | geçen süre: 0.4 sn | RAM: 4.01 GB

BAŞLADI: linear preprocessing — train fit_transform
TAMAMLANDI: linear preprocessing — train fit_transform | geçen süre: 21.1 sn | RAM: 4.44 GB
linear/train: (413378, 4068), nnz=174,858,894

BAŞLADI: linear preprocessing — validation transform
TAMAMLANDI: linear preprocessing — validation transform | geçen süre: 1.7 sn | RAM: 2.91 GB
linear/validation: (88581, 4068), nnz=37,468,890

BAŞLADI: linear preprocessing — test transform
TAMAMLANDI: linear preprocessing — test transform | geçen süre: 1.7 sn | RAM: 2.91 GB
linear/test: (88581, 4068), nnz=37,468,881
sklearn_tree cache bulunamadı; ortak Parquet'ten oluşturuluyor.

BAŞLADI: Ortak Parquet cache'ini yükleme
TAMAMLANDI: Ortak Parquet cache'ini yükleme | geçen süre: 0.3 sn | RAM: 6.49 GB

BAŞLADI: sklearn_tree preprocessing — train fit_transform
TAMAMLANDI: 

LightGBM train kategorileri öğreniliyor: 100%|██████████| 42/42 [00:01<00:00, 34.91 kolon/s]



BAŞLADI: lightgbm/train Parquet kaydı
TAMAMLANDI: lightgbm/train Parquet kaydı | geçen süre: 2.9 sn | RAM: 7.10 GB

BAŞLADI: lightgbm/validation Parquet kaydı
TAMAMLANDI: lightgbm/validation Parquet kaydı | geçen süre: 0.7 sn | RAM: 6.10 GB

BAŞLADI: lightgbm/test Parquet kaydı
TAMAMLANDI: lightgbm/test Parquet kaydı | geçen süre: 0.7 sn | RAM: 6.11 GB
Linear: (413378, 4068) (88581, 4068)
Tree: (413378, 4068) (88581, 4068)
LightGBM: (413378, 423) (88581, 423)


## 4. Ortak değerlendirme fonksiyonları

Training süresi, validation inference süresi, 10.000 satırlık tekrar ölçümü, model boyutu ve validation metrikleri aynı fonksiyonlarla hesaplanır.

In [ ]:
def ram_gb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)


def calculate_validation_metrics(y_true, probabilities, threshold=DIAGNOSTIC_THRESHOLD):
    predictions = (probabilities >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "diagnostic_threshold": float(threshold),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, probabilities)),
        "pr_auc": float(average_precision_score(y_true, probabilities)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "routed_rate_at_050": float(predictions.mean()),
    }


def measure_inference(model, X_validation):
    sample = X_validation[:min(INFERENCE_SAMPLE_SIZE, X_validation.shape[0])]
    _ = model.predict_proba(sample)[:, 1]
    durations = []
    for _ in tqdm(range(INFERENCE_REPEATS), desc="10K inference", leave=False):
        started = time.perf_counter()
        _ = model.predict_proba(sample)[:, 1]
        durations.append(time.perf_counter() - started)
    return float(np.mean(durations)), float(np.std(durations)), sample.shape[0]


def fit_and_evaluate(model_name, model_role, model, data, model_path):
    print("\n" + "=" * 78)
    print("MODEL:", model_name)
    print("=" * 78)
    ram_before = ram_gb()
    started = time.perf_counter()
    model.fit(data["X_train"], data["y_train"])
    training_seconds = time.perf_counter() - started
    ram_after = ram_gb()

    started = time.perf_counter()
    probabilities = model.predict_proba(data["X_validation"])[:, 1]
    validation_inference_seconds = time.perf_counter() - started
    inference_mean, inference_std, sample_size = measure_inference(
        model, data["X_validation"]
    )
    joblib.dump(model, model_path, compress=3)
    model_size_mb = model_path.stat().st_size / (1024 ** 2)

    metrics = calculate_validation_metrics(data["y_validation"], probabilities)
    metrics.update({
        "model_name": model_name,
        "model_role": model_role,
        "split_version": "common_v2",
        "preprocessing_version": "lazy_cache_v1",
        "imbalance_method": "class_weight_or_equivalent",
        "threshold_selected": False,
        "training_seconds": float(training_seconds),
        "validation_inference_seconds": float(validation_inference_seconds),
        "inference_seconds_10k_mean": inference_mean,
        "inference_seconds_10k_std": inference_std,
        "inference_sample_rows": sample_size,
        "model_size_mb": float(model_size_mb),
        "ram_before_gb": float(ram_before),
        "ram_after_fit_gb": float(ram_after),
    })
    print(pd.Series(metrics))
    return metrics, probabilities

## 5. Logistic Regression

In [15]:
logistic_model = LogisticRegression(
    solver="saga", max_iter=500, class_weight="balanced",
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1,
)
logistic_metrics, logistic_probabilities = fit_and_evaluate(
    "LogisticRegression", "light", logistic_model, linear_data,
    MODEL_DIR / "logistic_regression_light.joblib",
)


MODEL: LogisticRegression


c:\Users\pc\anaconda3\envs\torchcuda\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\pc\anaconda3\envs\torchcuda\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


max_iter reached after 486 seconds


TypeError: sparse array length is ambiguous; use getnnz() or shape[0]

## 6. Decision Tree

In [ ]:
decision_tree_model = DecisionTreeClassifier(
    max_depth=6, min_samples_leaf=50, class_weight="balanced",
    random_state=RANDOM_STATE,
)
decision_tree_metrics, decision_tree_probabilities = fit_and_evaluate(
    "DecisionTree", "light", decision_tree_model, tree_data,
    MODEL_DIR / "decision_tree_light.joblib",
)

## 7. Small Random Forest

In [ ]:
small_rf_model = RandomForestClassifier(
    n_estimators=50, max_depth=10, min_samples_leaf=50,
    max_features="sqrt", class_weight="balanced_subsample",
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1,
)
small_rf_metrics, small_rf_probabilities = fit_and_evaluate(
    "SmallRandomForest", "light", small_rf_model, tree_data,
    MODEL_DIR / "small_random_forest_light.joblib",
)

## 8. Small LightGBM

Early stopping validation ROC-AUC değerini izler; threshold seçimi yapılmaz.

In [ ]:
negative_count = int((lightgbm_data["y_train"] == 0).sum())
positive_count = int((lightgbm_data["y_train"] == 1).sum())
scale_pos_weight = negative_count / positive_count

small_lgbm_model = LGBMClassifier(
    objective="binary", n_estimators=300, learning_rate=0.05,
    max_depth=6, num_leaves=31, min_child_samples=100,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
)

print("\n" + "=" * 78)
print("MODEL: SmallLightGBM")
print("=" * 78)
ram_before = ram_gb()
started = time.perf_counter()
small_lgbm_model.fit(
    lightgbm_data["X_train"], lightgbm_data["y_train"],
    eval_set=[(lightgbm_data["X_validation"], lightgbm_data["y_validation"])],
    eval_metric="auc",
    callbacks=[
        __import__("lightgbm").early_stopping(50, verbose=True),
        __import__("lightgbm").log_evaluation(25),
    ],
)
training_seconds = time.perf_counter() - started
ram_after = ram_gb()
started = time.perf_counter()
small_lgbm_probabilities = small_lgbm_model.predict_proba(
    lightgbm_data["X_validation"]
)[:, 1]
validation_inference_seconds = time.perf_counter() - started
inference_mean, inference_std, sample_size = measure_inference(
    small_lgbm_model, lightgbm_data["X_validation"]
)
lgbm_path = MODEL_DIR / "small_lightgbm_light.joblib"
joblib.dump(small_lgbm_model, lgbm_path, compress=3)

small_lgbm_metrics = calculate_validation_metrics(
    lightgbm_data["y_validation"], small_lgbm_probabilities
)
small_lgbm_metrics.update({
    "model_name": "SmallLightGBM", "model_role": "light",
    "split_version": "common_v2", "preprocessing_version": "lazy_cache_v1",
    "imbalance_method": "scale_pos_weight", "threshold_selected": False,
    "training_seconds": float(training_seconds),
    "validation_inference_seconds": float(validation_inference_seconds),
    "inference_seconds_10k_mean": inference_mean,
    "inference_seconds_10k_std": inference_std,
    "inference_sample_rows": sample_size,
    "model_size_mb": lgbm_path.stat().st_size / (1024 ** 2),
    "ram_before_gb": float(ram_before), "ram_after_fit_gb": float(ram_after),
})
print(pd.Series(small_lgbm_metrics))

## 9. Validation karşılaştırma tablosu ve olasılık kaydı

Buradaki sıralama kesin seçim değildir. Görev 4'te validation threshold analizi tamamlandıktan sonra hafif model seçilecektir.

In [ ]:
comparison = pd.DataFrame([
    logistic_metrics, decision_tree_metrics, small_rf_metrics, small_lgbm_metrics,
]).sort_values(["pr_auc", "roc_auc"], ascending=False).reset_index(drop=True)

comparison_path = PATHS.metrics / "light_model_comparison_validation.csv"
comparison.to_csv(comparison_path, index=False)

validation_predictions = pd.DataFrame({
    "TransactionID": linear_data["id_validation"],
    "y_true": linear_data["y_validation"],
    "logistic_regression_probability": logistic_probabilities,
    "decision_tree_probability": decision_tree_probabilities,
    "small_random_forest_probability": small_rf_probabilities,
    "small_lightgbm_probability": small_lgbm_probabilities,
})
predictions_path = PREDICTIONS_DIR / "light_models_validation_predictions.parquet"
validation_predictions.to_parquet(predictions_path, index=False, compression="zstd")

metadata = {
    "experiment_stage": "task_2_light_model_comparison",
    "split_version": "common_v2",
    "test_used": False,
    "threshold_tuned": False,
    "diagnostic_threshold": DIAGNOSTIC_THRESHOLD,
    "models": comparison.to_dict(orient="records"),
}
metadata_path = PATHS.metadata / "light_model_comparison_metadata.json"
with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

display(comparison)
print("Karşılaştırma:", comparison_path.relative_to(PROJECT_ROOT))
print("Validation olasılıkları:", predictions_path.relative_to(PROJECT_ROOT))
print("Metadata:", metadata_path.relative_to(PROJECT_ROOT))

# Görev 2 tamamlanma koşulları

- Dört hafif model aynı train/validation splitinde çalıştırıldı.
- Test spliti yüklenmedi ve değerlendirilmedi.
- Threshold seçilmedi; `0.50` yalnızca tanısal karşılaştırma için kullanıldı.
- ROC-AUC, PR-AUC, precision, recall, F1, FP/FN, süre, RAM ve model boyutu kaydedildi.
- Validation olasılıkları Görev 4 threshold analizi için saklandı.

Sonraki görev, onaydan sonra ağır model karşılaştırmasıdır.